In [ ]:
# Install packages
!pip3 install transformers==4.45.2 sentencepiece==0.2.2 torch==2.5.1
!pip3 install datasets pandas
SEED = 7

Defaulting to user installation because normal site-packages is not writeable
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.


In [12]:
# import opus translation model
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

name = "Helsinki-NLP/opus-mt-nl-en"
tokenizer = AutoTokenizer.from_pretrained(name)
model = AutoModelForSeq2SeqLM.from_pretrained(name)

/Users/jaridzatromp/Library/Python/3.9/lib/python/site-packages/transformers/models/marian/tokenization_marian.py:175: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


In [13]:
from huggingface_hub import HfApi

api = HfApi()
for repo in ["Helsinki-NLP/opus-mt-nl-en",
             "yunvs/bart-base-wikilarge-simplification",
             "BramVanroy/ul2-base-dutch-simplification-mai-2023",
             "igornishka/dutch-municipal-sentence-simplification"]:
    print(repo, api.model_info(repo).sha)

Helsinki-NLP/opus-mt-nl-en 48af999f2c59b10c05ca6e008dcedc07677a9b15
yunvs/bart-base-wikilarge-simplification 050112d6e53fbde261fdc120ce19a687a9e13086
BramVanroy/ul2-base-dutch-simplification-mai-2023 4ed2e70657f452c3bed13f7c64f126220c3b6450


In [14]:
MODELS = {
    "mt":    ("Helsinki-NLP/opus-mt-nl-en", "48af999f2c59b10c05ca6e008dcedc07677a9b15"),
    "en_ts": ("yunvs/bart-base-wikilarge-simplification", "050112d6e53fbde261fdc120ce19a687a9e13086"),
    "nl_ts": ("BramVanroy/ul2-base-dutch-simplification-mai-2023", "050112d6e53fbde261fdc120ce19a687a9e13086"),
}

name, rev = MODELS["mt"]
tokenizer = AutoTokenizer.from_pretrained(name, revision=rev)
model = AutoModelForSeq2SeqLM.from_pretrained(name, revision=rev)

In [10]:
!pip3 list | grep torch

torch                         2.5.1
torchvision                   0.19.1
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.


In [ ]:
from datasets import load_dataset

ds = load_dataset(
    "igornishka/dutch-municipal-sentence-simplification",
    revision="<commit sha>",   # get it with HfApi().dataset_info(...).sha
)
df = ds["train"].to_pandas()
print(df.columns)   # ['Document ID', 'Complex', 'Simple']
print(len(df))

In [ ]:
# Load data from SimPA
import pandas as pd
from pathlib import Path

def read_lines(p):
    return Path(p).read_text(encoding="utf-8").splitlines()

d = "data/simpa/"

# Syntactic: 1 reference per sentence
ss = pd.DataFrame({"src": read_lines(d + "ss.original"),
                   "ss_ref": read_lines(d + "ss.simplified")})

# Lexical: 3 references per sentence, grouped by original
ls = pd.DataFrame({"src": read_lines(d + "ls.original"),
                   "ls_ref": read_lines(d + "ls.simplified")})
ls_refs = ls.groupby("src", sort=False)["ls_ref"].apply(list)

simpa = ss.join(ls_refs, on="src")
print(len(simpa), simpa["ls_ref"].str.len().value_counts())   # expect 1100, all 3

In [ ]:
# Dataset statistics
def stats(src, ref, name):
    s_len = src.str.split().str.len()
    r_len = ref.str.split().str.len()
    return {
        "dataset": name,
        "pairs": len(src),
        "src tokens/sent": round(s_len.mean(), 1),
        "ref tokens/sent": round(r_len.mean(), 1),
        "compression": round((r_len / s_len).mean(), 2),
        "% unchanged": round((src.str.strip() == ref.str.strip()).mean() * 100, 1),
    }

table = pd.DataFrame([
    stats(df["Complex"], df["Simple"], "Amsterdam (nl)"),
    stats(simpa["src"], simpa["ss_ref"], "SimPA (en, ss)"),
])
print(table.to_latex(index=False))   # paste straight into Overleaf